# Migrazioni, saldo migratorio e popolazione nata all'estero

Questo foglio separa i concetti: il saldo migratorio appartiene al bilancio demografico, i flussi annui descrivono ingressi e uscite, mentre paese di nascita e titolo di studio descrivono la composizione della popolazione residente.

Nota di lettura: i residenti nati in Italia senza cittadinanza italiana richiedono l'incrocio tra paese di nascita e cittadinanza. Le tavole aperte disponibili pubblicano queste due dimensioni come margini separati; per questo il notebook non stima quel sottoinsieme per differenza.

Fonti principali: [`demo_gind`](https://ec.europa.eu/eurostat/databrowser/view/demo_gind/default/table?lang=en), [`demo_r_gind3`](https://ec.europa.eu/eurostat/databrowser/view/demo_r_gind3/default/table?lang=en), [`migr_imm1ctz`](https://ec.europa.eu/eurostat/databrowser/product/view/migr_imm1ctz), [`migr_emi1ctz`](https://ec.europa.eu/eurostat/databrowser/product/view/migr_emi1ctz), [`migr_pop1ctz`](https://ec.europa.eu/eurostat/databrowser/view/migr_pop1ctz/default/table?lang=en), [`migr_pop3ctb`](https://ec.europa.eu/eurostat/databrowser/view/migr_pop3ctb/default/table?lang=en) ed [`edat_lfs_9917`](https://ec.europa.eu/eurostat/databrowser/product/view/edat_lfs_9917). Ogni grafico riporta fonte ed elaborazione in basso a sinistra.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from demografia.notebook_charts import (
    fig_births_deaths,
    fig_internal_migration_destinations,
    fig_kebab,
    fig_migrant_education_by_birth,
    fig_migrant_stock_categories,
    fig_migrant_tertiary_region,
    fig_migration,
    fig_migration_age_profile,
    fig_migration_citizenship_profile,
    fig_migration_destination_rank,
    fig_population_series,
    fig_regional_rank,
    fig_regional_series,
    load_notebook_tables,
    notebook_paths,
    parametri_disponibili,
)

paths = notebook_paths(ROOT)
tables = load_notebook_tables(paths["final"])


## Parametri

I valori sotto controllano l'intero foglio. Usare codici semplici: paesi in ISO3 (`ITA`, `ESP`), regioni NUTS2 (`ITC4`) e province NUTS3 (`ITC4C`). Per togliere il confronto usare `none`.


In [ ]:
PAESE = "ITA"  # Italia. Codice paese ISO3.
PAESE_CONFRONTO = "ESP"  # Spagna. Usare "none" per nessun confronto.

REGIONE = "ITC4"  # Lombardia. Codice NUTS2.
REGIONE_CONFRONTO = "ITI4"  # Lazio. Codice NUTS2.
PROVINCIA = "ITC4C"  # Milano. Codice NUTS3.
PROVINCIA_CONFRONTO = "ITC11"  # Torino. Codice NUTS3.

ANNO_FLUSSI = 2024  # Anno dei profili dei flussi migratori.
ANNO_NATI_ESTERO_RECENTE = 2025  # Anno recente dello stock di nati all'estero.
ANNO_NATI_ESTERO_STORICO = 2002  # Anno storico dello stock di nati all'estero.
ANNO_TERRITORI = 2024  # Anno usato per classifiche territoriali.
ANNO_ISTRUZIONE_MIGRANTI = 2024  # Anno della tavola LFS su istruzione e nascita.
AREA_ISTRUZIONE_MIGRANTI = "ITA"  # Italia. Usare codici della tabella sotto.
DETTAGLIO_STOCK = "countries"  # Opzioni: "countries" nazioni, "aggregates" aggregati, "all" entrambi.
MISURA_STOCK = "absolute"  # Opzioni: "absolute" migliaia di residenti, "percent_total" quota sul totale.
LIMITE_STOCK_NAZIONI = 25  # Numero massimo di nazioni o categorie nel grafico stock.

parametri_disponibili(tables, include_aree_istruzione_migranti=True)


## Saldo migratorio e componenti del bilancio

Questi grafici usano il bilancio demografico: immigrazione, emigrazione quando disponibile, saldo migratorio con aggiustamento statistico e variazione totale della popolazione. Servono per leggere il contributo delle migrazioni alla dinamica complessiva.


In [ ]:
fig_migration(tables, territory=PAESE, compare=PAESE_CONFRONTO).show()


In [ ]:
fig_population_series(tables, territory=PAESE, compare=PAESE_CONFRONTO, metric="net_migration_adjustment").show()


In [ ]:
fig_population_series(tables, territory=PAESE, compare=PAESE_CONFRONTO, metric="population_change").show()


## Età e cittadinanza dei flussi

Eurostat pubblica i flussi annui per età, sesso e cittadinanza. Qui la cittadinanza è usata come misura amministrativa vicina alla nazionalità, ma non coincide sempre con il paese di nascita. Le categorie estere sono distinte tra cittadini di altri paesi UE27 e cittadini extra UE27.


In [ ]:
fig_migration_age_profile(tables, flow="immigration", country=PAESE, compare=PAESE_CONFRONTO, year=ANNO_FLUSSI).show()


In [ ]:
fig_migration_age_profile(tables, flow="emigration", country=PAESE, compare=PAESE_CONFRONTO, year=ANNO_FLUSSI).show()


In [ ]:
fig_migration_citizenship_profile(tables, flow="immigration", country=PAESE, compare=PAESE_CONFRONTO, year=ANNO_FLUSSI).show()


In [ ]:
fig_migration_citizenship_profile(tables, flow="emigration", country=PAESE, compare=PAESE_CONFRONTO, year=ANNO_FLUSSI).show()


## Stock residente per nazione

Questa sezione guarda la popolazione residente, non i flussi annui. Il primo grafico usa il paese di nascita (`migr_pop3ctb`), il secondo la cittadinanza (`migr_pop1ctz`). Con `DETTAGLIO_STOCK = "countries"` vengono mostrate solo nazioni singole; gli aggregati Eurostat sono disponibili cambiando il parametro in `aggregates` o `all`.


In [ ]:
fig_migrant_stock_categories(
    tables,
    basis="country_of_birth",
    country=PAESE,
    compare=PAESE_CONFRONTO,
    year=ANNO_NATI_ESTERO_RECENTE,
    detail=DETTAGLIO_STOCK,
    measure=MISURA_STOCK,
    limit=LIMITE_STOCK_NAZIONI,
).show()


In [ ]:
fig_migrant_stock_categories(
    tables,
    basis="citizenship",
    country=PAESE,
    compare=PAESE_CONFRONTO,
    year=ANNO_NATI_ESTERO_RECENTE,
    detail=DETTAGLIO_STOCK,
    measure=MISURA_STOCK,
    limit=LIMITE_STOCK_NAZIONI,
).show()


## Destinazioni regionali e provinciali

Il dettaglio territoriale del bilancio consente di vedere dove si concentrano immigrazione, emigrazione e saldo migratorio. A livello regionale e provinciale la fonte non contiene sempre la stessa granularità per cittadinanza o titolo di studio: per questo la lettura territoriale resta separata dai profili dettagliati dei flussi nazionali.


In [ ]:
fig_migration_destination_rank(tables, level="region", metric="immigration", year=ANNO_TERRITORI, limit=25).show()


In [ ]:
fig_migration_destination_rank(tables, level="province", metric="immigration", year=ANNO_TERRITORI, limit=40).show()


In [ ]:
fig_regional_rank(tables, level="region", metric="net_migration_adjustment", year=ANNO_TERRITORI, limit=25).show()


In [ ]:
fig_regional_rank(tables, level="province", metric="net_migration_adjustment", year=ANNO_TERRITORI, limit=40).show()


In [ ]:
fig_regional_series(tables, focus=REGIONE, compare=REGIONE_CONFRONTO, metric="net_migration_adjustment").show()


In [ ]:
fig_regional_series(tables, focus=PROVINCIA, compare=PROVINCIA_CONFRONTO, metric="net_migration_adjustment").show()


## Titolo di studio e paese di nascita

Questa sezione usa Eurostat LFS: mostra quote percentuali della popolazione residente in famiglie private, per paese di nascita e regione NUTS2. Non è un flusso annuo di nuovi immigrati; è una fotografia dello stock residente. La quota di laureati è la categoria ISCED 5-8.


In [ ]:
fig_migrant_education_by_birth(tables, geo_code=AREA_ISTRUZIONE_MIGRANTI, year=ANNO_ISTRUZIONE_MIGRANTI).show()


In [ ]:
fig_migrant_tertiary_region(tables, country=PAESE, birth_group="FOR", year=ANNO_ISTRUZIONE_MIGRANTI, limit=21).show()


In [ ]:
fig_migrant_tertiary_region(tables, country=PAESE, birth_group="NAT", year=ANNO_ISTRUZIONE_MIGRANTI, limit=21).show()


## Migrazioni interne

Se la pipeline ufficiale ISTAT ha trovato una tabella origine-destinazione, questo grafico mostra le principali destinazioni dei trasferimenti interni di residenza. Se la tabella non è presente, la cella resta vuota ma il notebook continua a funzionare.


In [ ]:
fig_internal_migration_destinations(tables, year=ANNO_TERRITORI, limit=40).show()


## Popolazione nata all'estero

Il Kebab demografico qui usa lo stock dei residenti nati all'estero per età e sesso. È utile per confrontare la struttura anagrafica della popolazione immigrata con quella complessiva, ma non va letto come flusso annuale.


In [ ]:
fig_kebab(tables, territory=PAESE, year=ANNO_NATI_ESTERO_STORICO, population_kind="foreign_born").show()


In [ ]:
fig_kebab(tables, territory=PAESE, year=ANNO_NATI_ESTERO_RECENTE, population_kind="foreign_born").show()
